# Classical ML + CNN для SER

Признаки: **MFCC** (12 коэф. × 5 статистик), **Pitch/F0** (5 статистик),
**ZCR** (5 статистик), **DWT** (5 уровней × 3 статистики) → **85 признаков**.  

Модели: **SVM** (poly), **Random Forest**, **1D CNN**.  
Датасеты: **RESD** (7 классов) и **DUSHA** (5 классов) — обучаются **отдельно**, итого 6 моделей.

## 1. Install

In [ ]:
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'librosa', 'pywavelets', 'datasets', 'soundfile',
    'scikit-learn', 'matplotlib', 'seaborn', 'tqdm',
], check=True)
print('Done.')

## 2. Imports & shared config

In [ ]:
import os, warnings, pathlib
import numpy as np
import pandas as pd
import librosa
import pywt
import soundfile as sf
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from scipy.stats import skew, kurtosis
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    f1_score, classification_report, confusion_matrix,
)
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

SEED       = 42
SR_TARGET  = 16_000
N_MFCC     = 12
HOP_LENGTH = 512
N_FFT      = 2048
MAX_FRAMES = 216   # для CNN (≈ 6.9 с при hop=512)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 3. Feature extraction

In [ ]:
def _stats(arr):
    return [float(np.min(arr)), float(np.max(arr)),
            float(np.mean(arr)), float(np.median(arr)), float(np.std(arr))]

def extract_mfcc(wav):
    mfcc = librosa.feature.mfcc(y=wav, sr=SR_TARGET, n_mfcc=N_MFCC,
                                  n_fft=N_FFT, hop_length=HOP_LENGTH)
    return [v for coef in mfcc for v in _stats(coef)]  # 60

def extract_pitch(wav):
    min_lag = int(SR_TARGET / 500)
    max_lag = int(SR_TARGET / 50)
    f0s = []
    for start in range(0, max(1, len(wav) - 2048), HOP_LENGTH):
        frame = wav[start:start + 2048]
        if len(frame) < 2048:
            break
        corr = np.correlate(frame, frame, mode='full')[len(frame) - 1:]
        seg = corr[min_lag:min(max_lag, len(corr))]
        if len(seg) == 0:
            continue
        peak = np.argmax(seg) + min_lag
        f0s.append(SR_TARGET / peak if peak > 0 else 0.0)
    return _stats(np.array(f0s) if f0s else np.array([0.0]))  # 5

def extract_zcr(wav):
    zcr = librosa.feature.zero_crossing_rate(wav, hop_length=HOP_LENGTH)[0]
    return _stats(zcr)  # 5

def extract_dwt(wav):
    coeffs = pywt.wavedec(wav, 'db4', level=4)  # 5 массивов: cA4, cD4..cD1
    feats = []
    for c in coeffs:
        feats += [float(np.std(c)), float(skew(c)), float(kurtosis(c))]
    return feats  # 15

def extract_features(wav):
    return extract_mfcc(wav) + extract_pitch(wav) + extract_zcr(wav) + extract_dwt(wav)  # 85

def extract_mfcc_seq(wav, max_frames=MAX_FRAMES):
    """Для CNN: (max_frames, N_MFCC) с паддингом/обрезкой."""
    mfcc = librosa.feature.mfcc(y=wav, sr=SR_TARGET, n_mfcc=N_MFCC,
                                  n_fft=N_FFT, hop_length=HOP_LENGTH).T  # (T, 12)
    T = mfcc.shape[0]
    if T >= max_frames:
        return mfcc[:max_frames].astype(np.float32)
    return np.vstack([mfcc, np.zeros((max_frames - T, N_MFCC))]).astype(np.float32)

# имена для RF importance
feature_names = (
    [f'mfcc_{i:02d}_{s}' for i in range(N_MFCC) for s in ['min','max','mean','median','std']]
    + [f'pitch_{s}' for s in ['min','max','mean','median','std']]
    + [f'zcr_{s}'   for s in ['min','max','mean','median','std']]
    + [f'dwt_L{l}_{s}' for l in range(5) for s in ['std','skew','kurt']]
)
print(f'Feature vector: {len(feature_names)} dims')

## 4. Shared training helpers

In [ ]:
class SeqDataset(Dataset):
    def __init__(self, records):
        self.data = []
        for r in tqdm(records, desc='MFCC seq', leave=False):
            try:
                seq = extract_mfcc_seq(r['wav'])
                self.data.append((seq, r['label']))
            except Exception:
                pass
    def __len__(self):  return len(self.data)
    def __getitem__(self, i):
        x, y = self.data[i]
        return torch.tensor(x), torch.tensor(y, dtype=torch.long)


class AudioCNN(nn.Module):
    def __init__(self, num_classes, n_mfcc=N_MFCC, max_frames=MAX_FRAMES, dropout=0.25):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv1d(n_mfcc, 128, kernel_size=6, padding='same'), nn.ReLU(),
            nn.Conv1d(128, 128, kernel_size=5, padding='same'),    nn.ReLU(),
            nn.Dropout(dropout),
            nn.MaxPool1d(8),
        )
        self.block2 = nn.Sequential(
            nn.Conv1d(128, 128, kernel_size=5, padding='same'), nn.ReLU(),
            nn.Conv1d(128, 128, kernel_size=5, padding='same'), nn.ReLU(),
            nn.Conv1d(128, 128, kernel_size=5, padding='same'), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(128, 128, kernel_size=5, padding='same'), nn.ReLU(),
        )
        self.classifier = nn.Linear((max_frames // 8) * 128, num_classes)

    def forward(self, x):
        x = x.permute(0, 2, 1)   # (B,T,C) → (B,C,T)
        x = self.block1(x)
        x = self.block2(x)
        return self.classifier(x.flatten(1))


class EarlyStopping:
    def __init__(self, patience=10, path='best.pt'):
        self.patience = patience
        self.path     = path
        self.best     = -1.0
        self.counter  = 0
        self.stop     = False
    def step(self, metric, model):
        if metric > self.best:
            self.best = metric; self.counter = 0
            torch.save(model.state_dict(), self.path)
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True


def run_experiment(records, label_names, tag, out_dir='/kaggle/working'):
    """
    Полный эксперимент: SVM + RF + CNN на переданных записях.
    records: list of {'wav': np.array, 'label': int}
    label_names: list of str (индекс = id класса)
    tag: str ('resd' или 'dusha') — для имён файлов
    """
    out = pathlib.Path(out_dir)
    num_classes = len(label_names)

    # --- Features for SVM / RF ---
    print(f'\n[{tag.upper()}] Extracting flat features...')
    X_list, y_list = [], []
    for r in tqdm(records):
        try:
            X_list.append(extract_features(r['wav']))
            y_list.append(r['label'])
        except Exception:
            pass
    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.int64)

    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=y)
    scaler    = StandardScaler()
    X_tr_s    = scaler.fit_transform(X_tr)
    X_te_s    = scaler.transform(X_te)

    results = {}

    # --- SVM ---
    print(f'[{tag.upper()}] SVM...')
    svm = SVC(kernel='poly', degree=3, C=1.0, coef0=1.0,
              class_weight='balanced', random_state=SEED)
    svm.fit(X_tr_s, y_tr)
    svm_pred = svm.predict(X_te_s)
    results['SVM'] = _metrics(y_te, svm_pred)
    _print_metrics(f'{tag.upper()} SVM', results['SVM'])
    print(classification_report(y_te, svm_pred, target_names=label_names, zero_division=0))

    # --- Random Forest ---
    print(f'[{tag.upper()}] Random Forest...')
    rf = RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                 random_state=SEED, n_jobs=-1)
    rf.fit(X_tr_s, y_tr)
    rf_pred = rf.predict(X_te_s)
    results['RF'] = _metrics(y_te, rf_pred)
    _print_metrics(f'{tag.upper()} Random Forest', results['RF'])
    print(classification_report(y_te, rf_pred, target_names=label_names, zero_division=0))
    _plot_importance(rf, feature_names, tag, out)

    # --- CNN ---
    print(f'[{tag.upper()}] CNN...')
    rec_tr, rec_te = train_test_split(
        records, test_size=0.2, random_state=SEED,
        stratify=[r['label'] for r in records])
    tr_ds = SeqDataset(rec_tr)
    te_ds = SeqDataset(rec_te)
    tr_ld = DataLoader(tr_ds, batch_size=32, shuffle=True,  num_workers=2)
    te_ld = DataLoader(te_ds, batch_size=32, shuffle=False, num_workers=2)

    cnn  = AudioCNN(num_classes).to(device)
    crit = nn.CrossEntropyLoss()
    opt  = torch.optim.Adam(cnn.parameters(), lr=1e-3)
    sch  = torch.optim.lr_scheduler.ReduceLROnPlateau(
               opt, mode='max', factor=0.5, patience=5)
    es   = EarlyStopping(patience=10, path=str(out / f'best_cnn_{tag}.pt'))
    history = []

    for epoch in range(1, 61):
        cnn.train()
        loss_sum = 0.0
        for xb, yb in tr_ld:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            l = crit(cnn(xb), yb)
            l.backward(); opt.step()
            loss_sum += l.item() * len(yb)

        cnn.eval()
        ps, ls = [], []
        with torch.no_grad():
            for xb, yb in te_ld:
                ps.append(cnn(xb.to(device)).argmax(1).cpu().numpy())
                ls.append(yb.numpy())
        wacc = balanced_accuracy_score(np.concatenate(ls), np.concatenate(ps))
        lr   = opt.param_groups[0]['lr']
        history.append((loss_sum / len(tr_ds), wacc))
        sch.step(wacc); es.step(wacc, cnn)
        print(f'  Epoch {epoch:3d}  loss={loss_sum/len(tr_ds):.4f}  '
              f'wacc={wacc:.4f}  lr={lr:.2e}' + ('  *' if es.counter==0 else ''),
              flush=True)
        if es.stop:
            print(f'  Early stop at epoch {epoch}'); break

    cnn.load_state_dict(torch.load(str(out / f'best_cnn_{tag}.pt'), map_location=device))
    cnn.eval()
    ps, ls = [], []
    with torch.no_grad():
        for xb, yb in te_ld:
            ps.append(cnn(xb.to(device)).argmax(1).cpu().numpy())
            ls.append(yb.numpy())
    cnn_pred   = np.concatenate(ps)
    cnn_labels = np.concatenate(ls)
    results['CNN'] = _metrics(cnn_labels, cnn_pred)
    _print_metrics(f'{tag.upper()} CNN', results['CNN'])
    print(classification_report(cnn_labels, cnn_pred, target_names=label_names, zero_division=0))

    _plot_learning(history, tag, out)
    _plot_cm(cnn_labels, cnn_pred, label_names, tag, out)
    _plot_summary(results, tag, out)

    return results


# ── вспомогательные функции ──────────────────────────────────────────────────

def _metrics(y_true, y_pred):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'wacc':     balanced_accuracy_score(y_true, y_pred),
        'f1_macro': f1_score(y_true, y_pred, average='macro',    zero_division=0),
        'f1_w':     f1_score(y_true, y_pred, average='weighted', zero_division=0),
    }

def _print_metrics(title, m):
    print(f'\n=== {title} ===')
    for k, v in m.items():
        print(f'  {k:12s}: {v:.4f}')

def _plot_importance(rf, feat_names, tag, out):
    idx = np.argsort(rf.feature_importances_)[::-1][:30]
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh([feat_names[i] for i in idx[::-1]],
            rf.feature_importances_[idx[::-1]], color='steelblue')
    ax.set_xlabel('Importance')
    ax.set_title(f'[{tag.upper()}] RF — Top 30 Feature Importances')
    plt.tight_layout()
    plt.savefig(out / f'rf_importance_{tag}.png', dpi=150)
    plt.show()

def _plot_learning(history, tag, out):
    losses = [h[0] for h in history]
    waccs  = [h[1] for h in history]
    best   = max(waccs)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(losses); ax1.set_title(f'[{tag.upper()}] Train Loss'); ax1.set_xlabel('Epoch')
    ax2.plot(waccs, color='darkorange')
    ax2.axhline(best, color='red', linestyle='--', label=f'best={best:.4f}')
    ax2.set_title(f'[{tag.upper()}] Val Weighted Accuracy')
    ax2.set_xlabel('Epoch'); ax2.legend()
    plt.tight_layout()
    plt.savefig(out / f'cnn_curves_{tag}.png', dpi=150)
    plt.show()

def _plot_cm(labels, preds, label_names, tag, out):
    cm      = confusion_matrix(labels, preds)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.heatmap(cm,      annot=True, fmt='d',   cmap='Blues',
                xticklabels=label_names, yticklabels=label_names, ax=axes[0])
    axes[0].set_title(f'[{tag.upper()}] CNN Confusion Matrix (counts)')
    axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=label_names, yticklabels=label_names, ax=axes[1])
    axes[1].set_title(f'[{tag.upper()}] CNN Confusion Matrix (row-norm)')
    axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')
    plt.tight_layout()
    plt.savefig(out / f'cnn_cm_{tag}.png', dpi=150)
    plt.show()

def _plot_summary(results, tag, out):
    df = pd.DataFrame(results, index=['accuracy','wacc','f1_macro','f1_w']).T
    print(f'\n── [{tag.upper()}] Summary ──')
    print(df.to_string(float_format='{:.4f}'.format))
    df.to_csv(out / f'results_{tag}.csv')

print('Helpers ready.')

## 5. RESD — Load & Run (7 классов)

In [ ]:
from datasets import load_dataset

RESD_LABEL2ID = {
    'happiness': 0, 'sadness': 1, 'anger': 2,
    'fear': 3, 'disgust': 4, 'enthusiasm': 5, 'neutral': 6,
}
RESD_LABELS = ['happiness','sadness','anger','fear','disgust','enthusiasm','neutral']

ds = load_dataset('Aniemore/resd')
resd_records = []
for split in ['train', 'test']:
    for ex in tqdm(ds[split], desc=f'RESD {split}'):
        audio = ex['speech']
        wav   = np.array(audio['array'], dtype=np.float32)
        sr    = audio['sampling_rate']
        if sr != SR_TARGET:
            wav = librosa.resample(wav, orig_sr=sr, target_sr=SR_TARGET)
        resd_records.append({'wav': wav, 'label': RESD_LABEL2ID[ex['emotion']]})

from collections import Counter
cnt = Counter(r['label'] for r in resd_records)
print(f'Total: {len(resd_records)}')
for lid, n in sorted(cnt.items()):
    print(f'  {RESD_LABELS[lid]:12s}  {n}')

In [ ]:
resd_results = run_experiment(resd_records, RESD_LABELS, tag='resd')

## 6. DUSHA — Load & Run (5 классов)

In [ ]:
DUSHA_TSV       = '/kaggle/input/datasets/aleksandribryanov/agg-dusha/aggregated_majority.tsv'
DUSHA_AUDIO_DIR = '/kaggle/input/datasets/sigireddybalasai/dusha-datasetcrowd/crowd_train'
DUSHA_FRACTION  = 0.1   # 10% — ~10k записей, иначе слишком долго

DUSHA_LABEL2ID = {'neutral': 0, 'angry': 1, 'positive': 2, 'sad': 3, 'other': 4}
DUSHA_LABELS   = ['neutral', 'angry', 'positive', 'sad', 'other']

df = pd.read_csv(DUSHA_TSV, sep='\t')
df = df[df['aggregated_emo'].isin(DUSHA_LABEL2ID)]
df = df.sample(frac=DUSHA_FRACTION, random_state=SEED)
print(f'DUSHA subset: {len(df)} rows')

dusha_records = []
for _, row in tqdm(df.iterrows(), total=len(df), desc='DUSHA load'):
    path = pathlib.Path(DUSHA_AUDIO_DIR) / row['audio_path']
    if not path.exists():
        continue
    try:
        wav, sr = sf.read(str(path), dtype='float32')
        if wav.ndim > 1:
            wav = wav.mean(axis=1)
        if sr != SR_TARGET:
            wav = librosa.resample(wav, orig_sr=sr, target_sr=SR_TARGET)
        dusha_records.append({'wav': wav, 'label': DUSHA_LABEL2ID[row['aggregated_emo']]})
    except Exception:
        pass

cnt = Counter(r['label'] for r in dusha_records)
print(f'Loaded: {len(dusha_records)}')
for lid, n in sorted(cnt.items()):
    print(f'  {DUSHA_LABELS[lid]:12s}  {n}')

In [ ]:
dusha_results = run_experiment(dusha_records, DUSHA_LABELS, tag='dusha')

## 7. Итоговое сравнение всех 6 моделей

In [ ]:
rows = []
for model_name in ['SVM', 'RF', 'CNN']:
    for dataset, res in [('RESD', resd_results), ('DUSHA', dusha_results)]:
        m = res[model_name]
        rows.append({
            'Dataset': dataset, 'Model': model_name,
            'Accuracy': round(m['accuracy'], 4),
            'WAcc':     round(m['wacc'],     4),
            'F1 Macro': round(m['f1_macro'], 4),
            'F1 Weighted': round(m['f1_w'],  4),
        })

df_all = pd.DataFrame(rows).set_index(['Dataset', 'Model'])
print(df_all.to_string())
df_all.to_csv('/kaggle/working/results_all.csv')

# барчарт WAcc
fig, ax = plt.subplots(figsize=(9, 4))
x       = np.arange(3)
width   = 0.35
resd_w  = [resd_results[m]['wacc']  for m in ['SVM','RF','CNN']]
dusha_w = [dusha_results[m]['wacc'] for m in ['SVM','RF','CNN']]
ax.bar(x - width/2, resd_w,  width, label='RESD',  color='steelblue')
ax.bar(x + width/2, dusha_w, width, label='DUSHA', color='darkorange')
ax.set_xticks(x); ax.set_xticklabels(['SVM','RF','CNN'])
ax.set_ylabel('Weighted Accuracy')
ax.set_title('Weighted Accuracy — все 6 моделей')
ax.legend(); ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('/kaggle/working/results_comparison.png', dpi=150)
plt.show()